Juan Pablo Moran

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Cargamos el archivo y limpiamos datos

In [7]:
df = pd.read_csv("Heart Prediction Quantum Dataset.csv")

df["BloodPressure"] = df["BloodPressure"].str.replace(",", ".")
df["BloodPressure"] = pd.to_numeric(df["BloodPressure"])

genero_min = df["Gender"].str.lower()
genero_limpio = genero_min.str.rstrip(" {}")
genero_limpio[genero_limpio == "ombre"] = "hombre"
df["Gender"] = genero_limpio.replace({"hombre": 1, "mujer": 0}).astype(int)

df = df.drop_duplicates()
df = df.dropna()

X = df[["Age", "Gender", "BloodPressure", "Cholesterol", "HeartRate"]]
y = df["HeartDisease"]

In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

scaler = StandardScaler()
mlp = MLPClassifier(random_state=42)
model = make_pipeline(scaler, mlp)

Usamos el train test

In [4]:
from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, random_state=42)

Capa 1: de 5 a 15 neuronas
Capa 2: de 10 a 20 neuronas
Capa 3: de 5 a 10 neuronas

In [5]:
param_list = []
for i in range(5, 16):
    for j in range(10, 21):
        for k in range(5, 11):
            hidden = (int(i), int(j), int(k))
            param_list.append(hidden)
print(len(param_list))

726


In [6]:
from sklearn.model_selection import GridSearchCV
param_grid = {'mlpclassifier__hidden_layer_sizes': param_list}
grid = GridSearchCV(model, param_grid, cv=5, n_jobs=-1)

%time grid.fit(Xtrain, ytrain)
print(grid.best_params_)

CPU times: total: 3min 52s
Wall time: 4min 28s
{'mlpclassifier__hidden_layer_sizes': (12, 14, 5)}


C:\Users\juanm\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Ver qué tan bueno es ese modelo con los datos de prueba

In [10]:
model = grid.best_estimator_
yfit = model.predict(Xtest)
from sklearn.metrics import classification_report
print(classification_report(ytest, yfit))

              precision    recall  f1-score   support

           0       0.83      0.69      0.75        51
           1       0.81      0.91      0.85        74

    accuracy                           0.82       125
   macro avg       0.82      0.80      0.80       125
weighted avg       0.82      0.82      0.81       125

